This notebook create features based on the processed dataset from the previous notebook

In [1]:
import numpy as np
import pandas as pd

from toast_cap.utilities import config

In [2]:
run_date_str = '20230824'
s3_output = f's3://toast-datascience-sandbox/xxxxxxxx/{run_date_str}' # output directory

df = pd.read_parquet('s3://toast-datascience-sandbox/xxxxxxxx/full_raw_data.parquet') # read 'full_raw_data.parquet' generated from the previous notebook

# you can add some filters to reduce the dataset you're going to work with
df = df.loc[(~df['label_90'].isna()) & (df['dt'].dt.year>=2018)]
df = df.sort_values(by=['rid', 'dt'])

In [ ]:
### I created a lot of features below, you may add or remove some features

In [3]:
# time on Toast
df['years_with_toast'] = ((df['dt'] - pd.to_datetime(df['first_order_date'])).dt.days / 365).clip(lower=0)

In [4]:
# GPV trend
df['gpv_mean_365d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('365d')['totalCreditCardRevenue'].mean().values
df['gpv_mean_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('180d')['totalCreditCardRevenue'].mean().values
df['gpv_mean_90d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('90d')['totalCreditCardRevenue'].mean().values

## volatility
df['gpv_cv_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('180d')['totalCreditCardRevenue'].std().values / df['gpv_mean_180d'].values
df['gpv_cv_90d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('90d')['totalCreditCardRevenue'].std().values / df['gpv_mean_90d'].values

df['log_gpv'] = np.where(df['totalCreditCardRevenue']>0, np.log(df['totalCreditCardRevenue']+1), 0)
df['log_gpv_std_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('180d')['log_gpv'].std().values
df['log_gpv_std_90d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('90d')['log_gpv'].std().values
df = df.drop(columns='log_gpv')


df['gpv_mean_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['totalCreditCardRevenue'].mean().values

df['gpv_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['totalCreditCardRevenue'].median().values

df['gpv_median_84d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('84d')['totalCreditCardRevenue'].median().values

df['gpv_median_28d_median_84d_diff'] = df['gpv_median_28d'] - df['gpv_median_84d']

df['gpv_median_28d_mean_90d_diff'] = df['gpv_median_28d'] - df['gpv_mean_90d']

df['gpv_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(28).fillna(0).values
df['gpv_median_28d_84ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(84).fillna(0).values

/tmp/ipykernel_941/2348695909.py:16: RuntimeWarning: invalid value encountered in divide
  df['gpv_cv_180d'] = df.set_index('dt').groupby(
/tmp/ipykernel_941/2348695909.py:19: RuntimeWarning: invalid value encountered in divide
  df['gpv_cv_90d'] = df.set_index('dt').groupby(


In [5]:
# GPV per hour trend
df['gpv_per_hour'] = np.where(
    df['tx_hours'] == 0, 0,
    df['totalCreditCardRevenue'] / df['tx_hours']
)

df['gpv_per_hour_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['gpv_per_hour'].median().values

df['gpv_per_hour_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_per_hour_median_28d'].diff(28).fillna(0).values


In [6]:
# days with no GPV
df['flag_no_gpv'] = np.where(df['totalCreditCardRevenue']<=0, 1, 0)

df['days_no_gpv_180d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('180d')['flag_no_gpv'].sum().values
df['days_no_gpv_28d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('28d')['flag_no_gpv'].sum().values

df['days_no_gpv_28d_28ddiff'] = df.groupby(config.GROUP)['days_no_gpv_28d'].diff(28).fillna(0).values

df = df.drop(columns='flag_no_gpv')

In [7]:
# transaction hours
df['tx_hours_mean_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].mean().values
df['tx_hours_median_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].median().values

df['tx_hours_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['tx_hours'].median().values

df['tx_hours_median_28d_28ddiff'] = df.groupby(config.GROUP)['tx_hours_median_28d'].diff(28).fillna(0).values

In [8]:
# modules
df['has_oo_mod'] = np.where(
    df['live_online_ordering_module_count'] > 0, 1, 0
)
df['has_gc_mod'] = np.where(
    df['live_gift_card_module_count'] > 0, 1, 0
)

In [9]:
# 30-day no processing history
df['noprocessing_last_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('150d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

In [11]:
df.to_parquet(os.path.join(s3_output, 'processed_train.parquet'))